# Membership Inference Attack (MIA) Dataset 0

In [8]:
#import libraries
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
import os
print('Libraries imported!!')

Libraries imported!!


In [9]:
#define directory of functions and actual directory
FUNCTIONS_HOME = '../../../functions/evaluation_functions/' #home directory of the project
REAL_DATA_HOME = '../../../data/raw/rembrandt/' #home directory of the project
SYN_DATA_HOME  = '../../../data/processed/rembrandt/' #home directory of the project
FUNCTIONS_DIR = 'EVALUATION FUNCTIONS/PRIVACY'
ACTUAL_DIR = os.getcwd()

#change directory to functions directory
os.chdir(FUNCTIONS_HOME + FUNCTIONS_DIR)

#import functions for membership attack simulation
from membership_inference import evaluate_membership_attack

#change directory to actual directory
os.chdir(ACTUAL_DIR)
print('Functions imported!!')

Functions imported!!


## 1. Read real and synthetic datasets
In this part real and synthetic datasets are read.

In [10]:
#Define global variables
DATA_TYPES = ['Real','GM','ECDF','CTGAN','SDV', 'WGANGP']
SYNTHESIZERS = ['GM','ECDF','CTGAN','SDV','WGANGP']
FILEPATHS = {'Real' : REAL_DATA_HOME + '0_Depression_Data_Real_Train.csv',
            'GM' : SYN_DATA_HOME + '0_Depression_Data_Synthetic_GM.csv',
            'ECDF' : SYN_DATA_HOME + '0_Depression_Data_Synthetic_ECDF.csv',
            'SDV': SYN_DATA_HOME + '0_Depression_Data_Synthetic_SDV.csv',
            'CTGAN' : SYN_DATA_HOME + '0_Depression_Data_Synthetic_CTGAN.csv',
            'WGANGP' : SYN_DATA_HOME + '0_Depression_Data_Synthetic_WGANGP.csv'}
categorical_columns = ['group']
data = dict()
Q=5

In [11]:
#iterate over all datasets filepaths and read each dataset
data = dict()

for name, path in FILEPATHS.items() :
    data[name] = pd.read_csv(path)
    for col in categorical_columns :
        data[name][col] = data[name][col].astype('category').cat.codes
    numerical_columns = data[name].select_dtypes(include=['int64','float64']).columns.tolist()
    for col in numerical_columns :
        data[name][col] = pd.qcut(data[name][col], q=Q, duplicates='drop').cat.codes
data

{'Real':     group  ma1  hars1  ma2  hars2  ma3  hars3  ma4  hars4  ma5  ...  ma10  \
 0       0    0      0    0      0    0      0    0      0    0  ...     0   
 1       1    4      4    1      1    3      1    3      1    4  ...     3   
 2       1    2      2    0      0    1      2    2      2    1  ...     1   
 3       1    4      4    1      1    2      3    0      1    0  ...     1   
 4       1    4      4    3      4    3      4    4      4    4  ...     3   
 ..    ...  ...    ...  ...    ...  ...    ...  ...    ...  ...  ...   ...   
 91      0    0      1    1      3    0      1    0      0    2  ...     0   
 92      0    0      1    0      3    0      1    0      1    0  ...     0   
 93      0    0      0    0      0    1      0    2      0    0  ...     1   
 94      0    0      0    0      0    0      0    0      2    1  ...     0   
 95      1    0      1    1      0    2      2    1      2    2  ...     2   
 
     hars10  ma11  hars11  ma12  hars12  ma13  hars13 

In [ ]:
# Fix the random seed so the attacker-side shuffling below (and therefore
# the whole membership-inference simulation) is reproducible.
SEED = 42
np.random.seed(SEED)


In [12]:
#read TRAIN real dataset
train_data = pd.read_csv(REAL_DATA_HOME + '0_Depression_Data_Real_Train.csv')
for col in categorical_columns :
    train_data[col] = train_data[col].astype('category').cat.codes
for col in numerical_columns :
    train_data[col] = pd.qcut(train_data[col], q=Q, duplicates='drop').cat.codes
train_data = train_data.sample(frac=1)
    
#read TEST real dataset
test_data = pd.read_csv(REAL_DATA_HOME + '0_Depression_Data_Real_Test.csv')
for col in categorical_columns :
    test_data[col] = test_data[col].astype('category').cat.codes
for col in numerical_columns :
    test_data[col] = pd.qcut(test_data[col], q=Q, duplicates='drop').cat.codes
print(len(test_data))
test_data.index = range(len(train_data), len(train_data) + len(test_data))

real_data = (pd.concat([train_data[0:len(test_data)], test_data])).sample(frac=1)
real_data

24


,group,ma1,hars1,ma2,hars2,ma3,hars3,ma4,hars4,ma5,...,ma10,hars10,ma11,hars11,ma12,hars12,ma13,hars13,ma14,hars14
108,0,0,0,0,0,0,0,1,1,0,...,0,0,0,0,0,1,0,0,0,0
102,1,0,4,2,4,2,3,3,4,2,...,2,3,1,4,3,3,3,2,2,4
98,1,0,0,0,1,1,0,2,0,1,...,0,2,1,3,3,2,1,2,1,0
87,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
101,1,4,3,0,1,3,2,4,3,2,...,0,3,0,3,3,4,0,3,1,0
10,1,2,2,3,1,4,3,3,1,3,...,1,0,0,2,1,0,2,2,2,1
40,1,3,1,1,0,4,4,4,4,4,...,3,4,3,3,3,2,2,2,4,4
42,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
59,0,0,0,0,0,0,0,0,0,0,...,1,3,0,2,0,0,0,0,0,1
50,1,2,1,1,0,0,0,2,2,1,...,1,1,0,0,0,0,0,1,3,2


In [13]:
thresholds = [0.4, 0.3, 0.2, 0.1]
props = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1]

train_data_indexes = train_data.index.tolist()
precision_values_all = dict()
accuracy_values_all = dict()

for name in SYNTHESIZERS :
    print(name)
    precision_values = dict()
    accuracy_values = dict()
    
    for th in thresholds :
        precision_values[th] = []
        accuracy_values[th] = []
        
        for p in props :
            
            attacker_data = real_data.iloc[0:int(len(real_data)*p)]
            precision_vals, accuracy_vals = evaluate_membership_attack(attacker_data, train_data_indexes, data[name], th)
            
            precision_values[th].append(precision_vals)
            accuracy_values[th].append(accuracy_vals)
            
            print('Proportion ', p, ' Threshold ', th, ' analysed')
            print('- mean precision', np.mean(precision_values[th]))
            print('- mean accuracy', np.mean(accuracy_values[th]))
    print('###################################################')
    
    precision_values_all[name] = precision_values
    accuracy_values_all[name] = accuracy_values

GM
Proportion  0.2  Threshold  0.4  analysed
- mean precision 0.75
- mean accuracy 0.6666666666666666
Proportion  0.3  Threshold  0.4  analysed
- mean precision 0.775
- mean accuracy 0.6904761904761905
Proportion  0.4  Threshold  0.4  analysed
- mean precision 0.7547619047619047
- mean accuracy 0.6708437761069339
Proportion  0.5  Threshold  0.4  analysed
- mean precision 0.7535714285714286
- mean accuracy 0.6489661654135338
Proportion  0.6  Threshold  0.4  analysed
- mean precision 0.7361904761904762
- mean accuracy 0.6334586466165414
Proportion  0.7  Threshold  0.4  analysed
- mean precision 0.7246031746031746
- mean accuracy 0.6187912964228753
Proportion  0.8  Threshold  0.4  analysed
- mean precision 0.7210884353741497
- mean accuracy 0.6093399082120887
Proportion  0.9  Threshold  0.4  analysed
- mean precision 0.7218614718614719
- mean accuracy 0.6087538150344147
Proportion  1  Threshold  0.4  analysed
- mean precision 0.7157287157287158
- mean accuracy 0.6059293170676279
Proportio

In [ ]:
results_rows = []
for name in SYNTHESIZERS:
    for th in thresholds:
        for p, prec, acc in zip(props, precision_values_all[name][th], accuracy_values_all[name][th]):
            results_rows.append({
                'synthesizer': name,
                'threshold': th,
                'proportion': p,
                'precision': prec,
                'accuracy': acc
            })

mia_results = pd.DataFrame(results_rows)
mia_results.to_csv('INFERENCE TESTS RESULTS/membership_inference_results.csv', index=False)
mia_results
